## PostgreSQL User & Permissions Setup

For this assignment, you must create a dedicated database user and grant it the correct permissions, then create the database schema.

Run the given SQL file (database.sql) in pgAdmin (Query Tool).

Your Python code must connect using this user - as shown in class.

In the file you will see:
1. Create a database user.
_(You may choose a different username/password if you prefer.)_
2. Grant permissions.
3. Create the database schema.
4. Few queries for you to run and check if it works.

In [1]:
%pip install psycopg

Note: you may need to restart the kernel to use updated packages.


In [1]:
import psycopg
import os


## Database Schema

```sql
    CREATE TABLE students (
        student_id SERIAL PRIMARY KEY,
        name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL
    );

    CREATE TABLE enrollments (
        enrollment_id SERIAL PRIMARY KEY,
        student_id INTEGER REFERENCES students(student_id),
        course_name TEXT NOT NULL
        CONSTRAINT unique_student_course UNIQUE (student_id, course_name)
    );
```

# Part 1 – Basics

Write a Python function connect_db() that:

- Connects to a PostgreSQL database using credentials provided in environment variables
- Returns a connection object

In [5]:
# answer
def connect_db():
    """
    Returns a connection to the PostgreSQL database.
    """
    return psycopg.connect(
        host="localhost",
        dbname="postgres",
        user="demo",
        password="1234"
    )

In [7]:
conn = connect_db()

### Inserting and Querying Data 
Insert a Student 

In [10]:
# answer
def add_student(conn, name, email):
    """
    Inserts a new student into the students table.
    """
    try:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO students (name, email) VALUES (%s, %s)",
                (name, email)
            )
        conn.commit()
    except:
        conn.rollback()

    


Query Students 

In [13]:
# answer
def get_all_students(conn):
    """
    Returns a list of all students as tuples.
    """
    with conn.cursor() as cur:
        cur.execute(
            "SELECT * FROM students")
        return cur.fetchall()

Parameterized Queries 

In [16]:
def find_student_by_email(conn, email):
    """
    Returns the student record matching the given email,
    or None if no such student exists.
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT *
            FROM students
            WHERE email = %s
            """
            ,(email,)
        )
        res = cur.fetchone()
        if not res:
            return None
        return res

In [18]:
add_student(conn, "Name", "name@example.com")
print(get_all_students(conn))
print(find_student_by_email(conn, "name@example.com"))
print(find_student_by_email(conn, "does.not.exist@example.com"))

[(1, 'Alice Cohen', 'alice.cohen@example.com'), (2, 'Omar Haddad', 'omar.haddad@example.com'), (3, 'Noa Levi', 'noa.levi@example.com'), (4, 'Maya Rosen', 'maya.rosen@example.com'), (5, 'Daniel Kim', 'daniel.kim@example.com'), (6, 'Lina Saad', 'lina.saad@example.com'), (7, 'Eitan Bar', 'eitan.bar@example.com'), (8, 'Sara Aziz', 'sara.aziz@example.com'), (9, 'No Enroll Student', 'no.enroll@example.com'), (10, 'Name', 'name@example.com')]
(10, 'Name', 'name@example.com')
None


Create and populate a **courses** table

In [21]:
def create_courses_table(conn):
    """
    Create a table named courses if it does not already exist.

    Schema:
      - course_name TEXT PRIMARY KEY
      - credits INTEGER NOT NULL
    """
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS courses;")

        cur.execute(
            "CREATE TABLE courses (course_name TEXT PRIMARY KEY,credits INTEGER NOT NULL);"
        )
    conn.commit()

Hint: Use `ON CONFLICT DO NOTHING`.

In [24]:
def insert_courses(conn):
    """
    Insert the following rows into courses:

      ('Databases', 4)
      ('Algorithms', 3)
      ('Operating Systems', 4)
      ('Computer Networks', 3)

    The function must be safe to run multiple times.
    """
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                INSERT INTO courses (course_name,credits)
                VALUES
                ('Databases', 4),
                ('Algorithms', 3),
                ('Operating Systems', 4),
                ('Computer Networks', 3)
                ON CONFLICT (course_name) DO NOTHING;
                """
            )
        conn.commit()
    except:
        conn.rollback()


In [26]:
def get_all_courses(conn):
    """
    Return a list of (course_name, credits) tuples,
    sorted by course_name.
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT * FROM courses
            ORDER BY course_name
            """
            )
        return cur.fetchall()  
    

In [28]:
create_courses_table(conn)
insert_courses(conn)
print(get_all_courses(conn))

[('Algorithms', 3), ('Computer Networks', 3), ('Databases', 4), ('Operating Systems', 4)]


### Database Metadata 

Expected output example:

> ['student_id', 'name', 'email']

In [31]:
def get_table_columns(conn, table_name):
    """
    Returns a list of column names for the given table.

    Hint: use information_schema.columns
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT column_name
            FROM information_schema.columns
            WHERE table_name = %s
            """
            ,(table_name,)
        )
        rows = cur.fetchall()
    return [row[0] for row in rows]
    

### Error Handling 

In [34]:
def safe_add_student(conn, name, email):
    """
    Attempts to add a student.
    If the email already exists, handle the error gracefully
    and do not crash the program.
    """
    try:
        with conn.cursor() as cur:
            cur.execute(
                "INSERT INTO students (name, email) VALUES (%s, %s)",
                (name, email)
            )
        conn.commit()
    except:
        conn.rollback()
        print("student with the same email already in Students table.")

In [36]:
print(safe_add_student(conn, "X", "x@example.com"))
print(safe_add_student(conn, "Y", "x@example.com"))
print(get_table_columns(conn, "students"))
print(get_table_columns(conn, "enrollments"))
print(get_table_columns(conn, "courses"))

None
student with the same email already in Students table.
None
['student_id', 'name', 'email']
['enrollment_id', 'student_id', 'course_name']
['course_name', 'credits']


# Part 2 - SQL

Get a student’s enrollments

In [39]:
def get_student_courses(conn, email):
    """
    Given a student's email, return a list of course_name strings
    they are enrolled in, sorted alphabetically.
    If the student does not exist, return None.
    If the student exists but has no enrollments, return [].
    """
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT course_name
                FROM enrollments AS e
                JOIN students AS s
                ON e.student_id = s.student_id
                WHERE s.email = %s
                ORDER BY course_name
                """
                ,(email,)
            )
            
            c_names =  cur.fetchall()
            
            if not c_names:
                return None
                
            res = [c_name[0] for c_name in c_names]
        
        return res
        
    except:
        conn.rollback()
            


Get a course roster 

In [42]:
def get_course_roster(conn, course_name):
    """
    Return a list of (student_id, name, email) for students enrolled
    in the given course_name, sorted by student_id.
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT s.student_id, name, email
            FROM enrollments AS e
            JOIN students AS s
            ON e.student_id = s.student_id
            WHERE e.course_name = %s
            ORDER BY student_id
            """
            ,(course_name,)
        )
        
        c_names =  cur.fetchall()
        
        if not c_names:
            return []
            
        res = [(c_name[0],c_name[1],c_name[2]) for c_name in c_names]
    
    return res

Count students per course 

In [45]:
def count_students_per_course(conn):
    """
    Return a list of (course_name, num_students) sorted by num_students DESC,
    then course_name ASC.
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT course_name, COUNT(student_id) AS num_students
            FROM enrollments
            GROUP BY course_name
            ORDER BY num_students DESC , course_name
            """
        )
        
        c_names =  cur.fetchall()
        
        if not c_names:
            return None
            
        res = [(c_name[0],c_name[1]) for c_name in c_names]
    
    return res

Courses with at least N students

In [48]:
def popular_courses(conn, min_students):
    """
    Return course_name strings where the number of enrolled students >= min_students.
    Sorted alphabetically.
    """
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT course_name
            FROM enrollments
            GROUP BY course_name
            HAVING COUNT(student_id) >= %s
            ORDER BY course_name
            """
            ,(min_students,)
        )
        
        return [row[0] for row in cur.fetchall()]


Students with no enrollments

In [51]:
def students_without_enrollments(conn):
    """
    Return a list of (student_id, name, email) for students who are not enrolled
    in any course. Sorted by student_id.
    """
    try:
        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT s.student_id, name, email
                FROM students AS s
                LEFT JOIN enrollments AS e
                ON e.student_id = s.student_id
                WHERE e.course_name is NULL
                ORDER BY student_id
                """
            )
            
            c_names =  cur.fetchall()
            
            if not c_names:
                return None
                
            res = [(c_name[0],c_name[1],c_name[2]) for c_name in c_names]
        
        return res

    except:
        conn.rollback()

In [53]:
print(get_student_courses(conn, "alice.cohen@example.com"))
print(get_course_roster(conn, "Databases"))
print(get_course_roster(conn, "Algorithms"))
print(count_students_per_course(conn))
print(popular_courses(conn, 3))
print(students_without_enrollments(conn))

['Databases', 'Operating Systems']
[(1, 'Alice Cohen', 'alice.cohen@example.com'), (2, 'Omar Haddad', 'omar.haddad@example.com'), (3, 'Noa Levi', 'noa.levi@example.com'), (5, 'Daniel Kim', 'daniel.kim@example.com'), (6, 'Lina Saad', 'lina.saad@example.com'), (8, 'Sara Aziz', 'sara.aziz@example.com')]
[(5, 'Daniel Kim', 'daniel.kim@example.com')]
[('Databases', 6), ('Algorithms', 1), ('Computer Networks', 1), ('Discrete Math', 1), ('Machine Learning', 1), ('Operating Systems', 1)]
['Databases']
[(9, 'No Enroll Student', 'no.enroll@example.com'), (10, 'Name', 'name@example.com'), (11, 'X', 'x@example.com')]


Enroll a student

In [56]:
def enroll_by_email(conn, email, course_name):
    """
    Enroll the student with the given email into course_name.

    Return:
      - True if enrollment was added
      - False if the student doesn't exist
      - False if already enrolled in that course

    Must be safe and atomic:
      - Use a transaction
      - Rollback on any error
    """
    try:
        with conn.cursor() as cur:
            
            s_to_insert = find_student_by_email(conn, email)
            if not s_to_insert:
                return False
            student_id = s_to_insert[0]
            
            cur.execute("SELECT 1 FROM enrollments WHERE student_id = %s AND course_name = %s", 
            (student_id, course_name))
            
            if cur.fetchall():
                return False
            cur.execute(
                "INSERT INTO enrollments (student_id, course_name) VALUES (%s, %s)"
                ,(s_to_insert[0], course_name)
            )
        conn.commit()
        return True

    except Exception as e:
        conn.rollback()
        return False


In [58]:
print(get_course_roster(conn, "Algorithms"))
print(enroll_by_email(conn, "omar.haddad@example.com", "Algorithms"))
print(get_course_roster(conn, "Algorithms"))

[(5, 'Daniel Kim', 'daniel.kim@example.com')]
True
[(2, 'Omar Haddad', 'omar.haddad@example.com'), (5, 'Daniel Kim', 'daniel.kim@example.com')]
